# Notebook 07 — Anomaly Vectorization Comparison

This notebook compares two anomaly feature spaces to understand how vector representation affects anomaly ranking:

- **Model A**: Sparse character TF-IDF + structural features
- **Model B**: Dense TF-IDF + LSA + structural features

Goal:

- Check whether LSA compression helps or hurts anomaly ranking quality.
- Measure overlap between top anomaly sets produced by each model.
- Export side-by-side diagnostics for report discussion.

Reader guide:

- Both models use the same normalized input text; only the vector representation changes.
- Ranking outputs are saved to `data/results/`.
- Spearman rank correlation and top-k overlap are used to quantify agreement.

## Setup

Add the `src/` directory to the Python path so that project modules are importable without installing the package.

In [ ]:
import sys
from pathlib import Path

project_root_path = Path.cwd().parent
if str(project_root_path / "src") not in sys.path:
    sys.path.insert(0, str(project_root_path / "src"))

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import issparse
from scipy.stats import spearmanr

from anomaly_detection import TextAnomalyDetector
from core.data_io import ArticleDataset
from exploration import attach_original_text_by_doc_id, build_anomaly_candidate_table
from preprocessing import StructuralFeatureExtractor, TextNormalizer, TextPreprocessor


## Load and normalize data


In [2]:
project_root_path = Path.cwd().parent
results_data_directory_path = project_root_path / "data" / "results"
results_data_directory_path.mkdir(parents=True, exist_ok=True)

articles_csv_path = project_root_path / "data" / "raw" / "articles.csv"
articles_data_frame = ArticleDataset(input_csv_path=articles_csv_path).load_articles()

document_ids = articles_data_frame["doc_id"].tolist()
raw_texts = articles_data_frame["text"].tolist()
normalized_bundle = TextNormalizer().normalize_for_both_tasks(raw_texts)

structural_feature_extractor = StructuralFeatureExtractor()
structural_feature_matrix = structural_feature_extractor.transform(raw_texts)
structural_feature_matrix.shape


(2164, 15)

## Model A: Sparse TF-IDF (char n-grams) + structural features

This model keeps a sparse lexical view and adds dense structural signals.


In [3]:
sparse_preprocessor = TextPreprocessor(
    vectorization_model_name="tfidf",
    max_features=30000,
    min_document_frequency=1,
    max_document_frequency=1.0,
    ngram_range=(3, 5),
    analyzer_mode="char_wb",
)

sparse_features = sparse_preprocessor.fit_transform(normalized_bundle.anomaly_texts)
if issparse(sparse_features):
    sparse_dense_features = np.asarray(sparse_features.todense(), dtype=np.float64)
else:
    sparse_dense_features = np.asarray(sparse_features, dtype=np.float64)

sparse_hybrid_features = np.hstack([sparse_dense_features, structural_feature_matrix])

sparse_detector = TextAnomalyDetector(contamination_ratio=0.02, random_seed=42)
sparse_mask, sparse_scores = sparse_detector.run_detection(sparse_hybrid_features)

sparse_table = build_anomaly_candidate_table(
    document_ids=document_ids,
    anomaly_scores=sparse_scores.tolist(),
    anomaly_mask=sparse_mask.tolist(),
)
sparse_table_with_text = attach_original_text_by_doc_id(
    anomaly_table=sparse_table,
    document_ids=document_ids,
    raw_texts=raw_texts,
)
sparse_table_with_text.to_csv(results_data_directory_path / "notebook_07_sparse_hybrid_anomaly_candidates.csv", index=False)

sparse_table_with_text.head(20)


,doc_id,score,predicted_anomaly,anomaly_rank,text
0,DOC_01716,-0.399055,True,1,Individual leaders by total points (Final stan...
1,DOC_01037,-0.392773,True,2,The FLYERS blew a 3-0 lead over the Buffalo Sa...
2,DOC_01877,-0.391717,True,3,"ETHER IMPLODES 2 EARTH CORE, IS GRAVITY!!! Thi..."
3,DOC_00228,-0.375834,True,4,I've been to three talks in the last month whi...
4,DOC_01260,-0.375700,True,5,"Sorry I missed you Raymond, I was just out in ..."
5,DOC_01607,-0.371927,True,6,McDonnell Douglas rolls out DC-X HUNTINGTON BE...
6,DOC_01706,-0.370645,True,7,"[Newsgroups: m.h.a added, followups set to mos..."
7,DOC_00873,-0.369118,True,8,"Apparently, Part 2 (defensemen numbered 2 thro..."
8,DOC_00204,-0.366947,True,9,I may not be the world's greatest expert on ch...
9,DOC_00534,-0.366191,True,10,Here are the standings after the April 6 updat...


## Model B: Dense TF-IDF + LSA + structural features

This model uses dense semantic compression and the same structural signals.


In [4]:
lsa_preprocessor = TextPreprocessor(
    vectorization_model_name="tfidf_lsa_dense",
    max_features=30000,
    min_document_frequency=1,
    max_document_frequency=1.0,
    ngram_range=(3, 5),
    analyzer_mode="char_wb",
    dense_embedding_dimension=256,
    random_seed=42,
)

lsa_features = lsa_preprocessor.fit_transform(normalized_bundle.anomaly_texts)
lsa_dense_features = np.asarray(lsa_features, dtype=np.float64)
lsa_hybrid_features = np.hstack([lsa_dense_features, structural_feature_matrix])

lsa_detector = TextAnomalyDetector(contamination_ratio=0.02, random_seed=42)
lsa_mask, lsa_scores = lsa_detector.run_detection(lsa_hybrid_features)

lsa_table = build_anomaly_candidate_table(
    document_ids=document_ids,
    anomaly_scores=lsa_scores.tolist(),
    anomaly_mask=lsa_mask.tolist(),
)
lsa_table_with_text = attach_original_text_by_doc_id(
    anomaly_table=lsa_table,
    document_ids=document_ids,
    raw_texts=raw_texts,
)
lsa_table_with_text.to_csv(results_data_directory_path / "notebook_07_lsa_hybrid_anomaly_candidates.csv", index=False)

lsa_table_with_text.head(20)


,doc_id,score,predicted_anomaly,anomaly_rank,text
0,DOC_01449,-0.508901,True,1,<>Hismanal (astemizole) is most definitely lin...
1,DOC_01990,-0.492500,True,2,3rd uptade: Here are the standings for the pol...
2,DOC_00829,-0.488962,True,3,2nd uptade: Here are the standings for the pol...
3,DOC_01509,-0.488149,True,4,I am scanning in a color image and it looks fi...
4,DOC_01885,-0.487702,True,5,Hi guys. I am scanning in a color image and it...
5,DOC_00508,-0.483067,True,6,==============================================...
6,DOC_01373,-0.482354,True,7,Hismanal (astemizole) is most definitely linke...
7,DOC_01851,-0.480838,True,8,All you have to do is turn it in to the police...
8,DOC_01064,-0.480297,True,9,: I have one complaint for the cameramen doing...
9,DOC_00909,-0.479055,True,10,Les Bartel's comments: Let me add my .02 in. I...


## Compare top anomalies (top-20 side by side)


In [5]:
raw_text_lookup_by_doc_id = {document_id: text for document_id, text in zip(document_ids, raw_texts, strict=False)}
comparison_table = pd.DataFrame(
    {
        "rank": range(1, 21),
        "sparse_doc_id": sparse_table.head(20)["doc_id"].tolist(),
        "sparse_text": [
            str(raw_text_lookup_by_doc_id.get(document_id, ""))
            for document_id in sparse_table.head(20)["doc_id"].astype(str).tolist()
        ],
        "lsa_doc_id": lsa_table.head(20)["doc_id"].tolist(),
        "lsa_text": [
            str(raw_text_lookup_by_doc_id.get(document_id, ""))
            for document_id in lsa_table.head(20)["doc_id"].astype(str).tolist()
        ],
        "sparse_score": sparse_table.head(20)["score"].tolist(),
        "lsa_score": lsa_table.head(20)["score"].tolist(),
    }
)
comparison_table.to_csv(results_data_directory_path / "notebook_07_top20_comparison.csv", index=False)
comparison_table


,rank,sparse_doc_id,sparse_text,lsa_doc_id,lsa_text,sparse_score,lsa_score
0,1,DOC_01716,Individual leaders by total points (Final stan...,DOC_01449,<>Hismanal (astemizole) is most definitely lin...,-0.399055,-0.508901
1,2,DOC_01037,The FLYERS blew a 3-0 lead over the Buffalo Sa...,DOC_01990,3rd uptade: Here are the standings for the pol...,-0.392773,-0.492500
2,3,DOC_01877,"ETHER IMPLODES 2 EARTH CORE, IS GRAVITY!!! Thi...",DOC_00829,2nd uptade: Here are the standings for the pol...,-0.391717,-0.488962
3,4,DOC_00228,I've been to three talks in the last month whi...,DOC_01509,I am scanning in a color image and it looks fi...,-0.375834,-0.488149
4,5,DOC_01260,"Sorry I missed you Raymond, I was just out in ...",DOC_01885,Hi guys. I am scanning in a color image and it...,-0.375700,-0.487702
5,6,DOC_01607,McDonnell Douglas rolls out DC-X HUNTINGTON BE...,DOC_00508,==============================================...,-0.371927,-0.483067
6,7,DOC_01706,"[Newsgroups: m.h.a added, followups set to mos...",DOC_01373,Hismanal (astemizole) is most definitely linke...,-0.370645,-0.482354
7,8,DOC_00873,"Apparently, Part 2 (defensemen numbered 2 thro...",DOC_01851,All you have to do is turn it in to the police...,-0.369118,-0.480838
8,9,DOC_00204,I may not be the world's greatest expert on ch...,DOC_01064,: I have one complaint for the cameramen doing...,-0.366947,-0.480297
9,10,DOC_00534,Here are the standings after the April 6 updat...,DOC_00909,Les Bartel's comments: Let me add my .02 in. I...,-0.366191,-0.479055


## Compare overlap diagnostics


In [6]:
def compute_top_k_overlap(
    sparse_ranked_table: pd.DataFrame,
    lsa_ranked_table: pd.DataFrame,
    top_k: int,
) -> int:
    """Calculates overlap between two ranked top-k anomaly sets."""
    sparse_top_ids = set(sparse_ranked_table.head(top_k)["doc_id"].tolist())
    lsa_top_ids = set(lsa_ranked_table.head(top_k)["doc_id"].tolist())
    return len(sparse_top_ids.intersection(lsa_top_ids))


overlap_diagnostics = pd.DataFrame(
    {
        "top_k": [10, 25, 50, 75, 100],
        "overlap_count": [
            compute_top_k_overlap(sparse_table, lsa_table, top_k_value) for top_k_value in [10, 25, 50, 75, 100]
        ],
    }
)
overlap_diagnostics.to_csv(results_data_directory_path / "notebook_07_overlap_diagnostics.csv", index=False)
overlap_diagnostics


,top_k,overlap_count
0,10,0
1,25,4
2,50,4
3,75,6
4,100,8


## Why overlap can be low


In [7]:
spearman_result = spearmanr(sparse_scores, lsa_scores)
score_rank_correlation = float(spearman_result.statistic)

summary_diagnostics = {
    "sparse_predicted_anomaly_count": int(sparse_mask.sum()),
    "lsa_predicted_anomaly_count": int(lsa_mask.sum()),
    "spearman_score_correlation": float(score_rank_correlation),
}
summary_diagnostics_data_frame = pd.DataFrame([summary_diagnostics])
summary_diagnostics_data_frame.to_csv(
    results_data_directory_path / "notebook_07_summary_diagnostics.csv",
    index=False,
)
summary_diagnostics


{'sparse_predicted_anomaly_count': 44,
 'lsa_predicted_anomaly_count': 44,
 'spearman_score_correlation': -0.1315260045935268}

## Overlap score in top-50


In [8]:
sparse_top_50 = set(sparse_table.head(50)["doc_id"].tolist())
lsa_top_50 = set(lsa_table.head(50)["doc_id"].tolist())
overlap_count = len(sparse_top_50.intersection(lsa_top_50))

pd.DataFrame([{"top_50_overlap_count": overlap_count}]).to_csv(
    results_data_directory_path / "notebook_07_top50_overlap.csv",
    index=False,
)
overlap_count


4

## Deterministic top-50 export candidate

This export uses score ranking from the dense hybrid model.
It gives exactly 50 ids for assignment-style output.


In [9]:
lsa_top_50_candidate = lsa_table.head(50).copy()
lsa_top_50_candidate["anomaly"] = lsa_top_50_candidate.index + 1

anomaly_candidate_output = lsa_top_50_candidate[["anomaly", "doc_id"]]
anomaly_candidate_output_with_text = attach_original_text_by_doc_id(
    anomaly_table=anomaly_candidate_output,
    document_ids=document_ids,
    raw_texts=raw_texts,
)
anomaly_candidate_output_with_text.to_csv(results_data_directory_path / "notebook_07_anomalies_candidate.csv", index=False)

anomaly_candidate_output_with_text.head(10)


,anomaly,doc_id,text
0,1,DOC_01449,<>Hismanal (astemizole) is most definitely lin...
1,2,DOC_01990,3rd uptade: Here are the standings for the pol...
2,3,DOC_00829,2nd uptade: Here are the standings for the pol...
3,4,DOC_01509,I am scanning in a color image and it looks fi...
4,5,DOC_01885,Hi guys. I am scanning in a color image and it...
5,6,DOC_00508,==============================================...
6,7,DOC_01373,Hismanal (astemizole) is most definitely linke...
7,8,DOC_01851,All you have to do is turn it in to the police...
8,9,DOC_01064,: I have one complaint for the cameramen doing...
9,10,DOC_00909,Les Bartel's comments: Let me add my .02 in. I...


## Files produced by this notebook

- `data/results/notebook_07_sparse_hybrid_anomaly_candidates.csv`
- `data/results/notebook_07_lsa_hybrid_anomaly_candidates.csv`
- `data/results/notebook_07_top20_comparison.csv`
- `data/results/notebook_07_overlap_diagnostics.csv`
- `data/results/notebook_07_summary_diagnostics.csv`
- `data/results/notebook_07_top50_overlap.csv`
- `data/results/notebook_07_anomalies_candidate.csv`
